# Matching Logic for ProMatch

Find the players most similar to a user's attributes using k-nearest-neighbours (scikit-learn).

## 1. Load the players

In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

players = pd.read_csv("players.csv")
players.head()

,name,club,nationality,age,overall,positions,pace,shooting,passing,dribbling,defending,physical
0,L. Messi,Paris Saint-Germain,Argentina,34,93,"RW, ST, CF",85,92,91,95,34,65
1,R. Lewandowski,FC Bayern München,Poland,32,92,ST,78,92,79,86,44,82
2,Cristiano Ronaldo,Manchester United,Portugal,36,91,"ST, LW",87,94,80,88,34,75
3,Neymar Jr,Paris Saint-Germain,Brazil,29,91,"LW, CAM",91,83,86,94,37,63
4,K. De Bruyne,Manchester City,Belgium,30,91,"CM, CAM",76,86,93,88,64,78


## 2. The six attributes I match on

In [2]:
attributes = ["pace", "shooting", "passing", "dribbling", "defending", "physical"]
players[attributes].head()

,pace,shooting,passing,dribbling,defending,physical
0,85,92,91,95,34,65
1,78,92,79,86,44,82
2,87,94,80,88,34,75
3,91,83,86,94,37,63
4,76,86,93,88,64,78


## 3. Example user input

The six values a user would type into the form.

In [3]:
user = {
    "pace": 80,
    "shooting": 70,
    "passing": 75,
    "dribbling": 82,
    "defending": 40,
    "physical": 65,
}

# put the values in the same order as the attributes list
user_vector = np.array([user[a] for a in attributes])
user_vector

array([80, 70, 75, 82, 40, 65])

## 4. Find similar players with k-NN

scikit-learn's NearestNeighbors finds the closest players. Euclidean = overall similarity.

In [4]:
knn = NearestNeighbors(n_neighbors=10, metric="euclidean")
knn.fit(players[attributes].to_numpy())

distances, indices = knn.kneighbors([user_vector])
players.iloc[indices[0]][["name", "club", "positions"] + attributes]

,name,club,positions,pace,shooting,passing,dribbling,defending,physical
1236,R. Orsolini,Bologna,"CAM, RW",80,70,74,81,39,65
912,Matheus Pereira,Al Hilal,"CAM, RM, RW",79,74,75,80,39,63
1925,F. Pizzini,Defensa y Justicia,RM,78,69,72,78,40,65
1618,M. Pjaca,Torino F.C.,"LW, RW, CAM",76,67,74,81,41,63
1654,R. Del Castillo,Stade Brestois 29,"RM, LM",77,68,72,79,39,65
1501,F. Di Francesco,Empoli,"CF, LW, ST",79,70,73,81,39,60
716,Guilson Paiva,Flamengo,"RM, RW",83,69,75,78,42,68
1589,Rúben Lameiras,Vitória de Guimarães,"RW, LW",76,71,72,79,42,64
725,J. Harrison,Leeds United,"LM, RM",83,71,72,79,42,68
807,A. Townsend,Everton,"RM, RW",77,74,75,79,38,67


## 5. Match %

Turn the distance into an easy 0-100 score (closer = higher).

In [5]:
# the biggest possible distance (all six attributes 99 apart)
max_distance = np.sqrt(len(attributes) * (99 ** 2))

results = players.iloc[indices[0]].copy()
results["match_percent"] = ((1 - distances[0] / max_distance) * 100).round(1)
results[["name", "club", "match_percent"]]

,name,club,match_percent
1236,R. Orsolini,Bologna,99.3
912,Matheus Pereira,Al Hilal,97.9
1925,F. Pizzini,Defensa y Justicia,97.7
1618,M. Pjaca,Torino F.C.,97.7
1654,R. Del Castillo,Stade Brestois 29,97.7
1501,F. Di Francesco,Empoli,97.7
716,Guilson Paiva,Flamengo,97.4
1589,Rúben Lameiras,Vitória de Guimarães,97.4
725,J. Harrison,Leeds United,97.4
807,A. Townsend,Everton,97.3


## 6. Matching by playing style

Switch the metric to cosine to match the shape of the attributes, not the overall level.

In [6]:
style_knn = NearestNeighbors(n_neighbors=10, metric="cosine")
style_knn.fit(players[attributes].to_numpy())

style_dist, style_idx = style_knn.kneighbors([user_vector])
players.iloc[style_idx[0]][["name", "club", "positions"] + attributes]

,name,club,positions,pace,shooting,passing,dribbling,defending,physical
1236,R. Orsolini,Bologna,"CAM, RW",80,70,74,81,39,65
1654,R. Del Castillo,Stade Brestois 29,"RM, LM",77,68,72,79,39,65
1925,F. Pizzini,Defensa y Justicia,RM,78,69,72,78,40,65
9055,S. Maier,Türkgücü München,"CAM, CM, LM",67,61,65,68,34,55
5080,D. Fagundez,Austin FC,"CAM, CM, LM",73,65,68,74,39,61
6283,Matheus Pereira,FC Barcelona,"CAM, RM",69,60,65,74,35,58
1618,M. Pjaca,Torino F.C.,"LW, RW, CAM",76,67,74,81,41,63
298,J. Correa,Inter,"CF, ST, CAM",84,75,77,85,39,69
12110,L. Damer,TSV Havelse,"CAM, ST",67,58,63,67,36,53
2687,Paulinho,Bayer 04 Leverkusen,"CAM, LW, RW",77,67,69,80,36,61


## 7. Adding weights

Multiply an attribute to make it matter more. Here pace and dribbling count double.

In [7]:
weights = {"pace": 2, "shooting": 1, "passing": 1, "dribbling": 2, "defending": 1, "physical": 1}
weight_vector = np.array([weights[a] for a in attributes])

weighted_knn = NearestNeighbors(n_neighbors=10, metric="euclidean")
weighted_knn.fit(players[attributes].to_numpy() * weight_vector)

w_dist, w_idx = weighted_knn.kneighbors([user_vector * weight_vector])
players.iloc[w_idx[0]][["name", "club"] + attributes]

,name,club,pace,shooting,passing,dribbling,defending,physical
1236,R. Orsolini,Bologna,80,70,74,81,39,65
1501,F. Di Francesco,Empoli,79,70,73,81,39,60
912,Matheus Pereira,Al Hilal,79,74,75,80,39,63
841,B. Traoré,Aston Villa,78,73,73,82,45,66
707,Nuno Santos,Sporting CP,81,75,75,80,43,67
1289,Trincão,Wolverhampton Wanderers,79,71,70,81,37,69
982,C. Hudson-Odoi,Chelsea,82,67,74,81,40,59
1605,T. Minamino,Liverpool,79,73,69,81,36,63
874,T. Bongonda,KRC Genk,80,71,72,82,33,61
349,A. Sánchez,Inter,78,76,78,83,43,68


## 8. One function for everything

Combines the metric choice and weights. I will use this in the backend next.

In [8]:
def recommend(user, method="euclidean", weights=None, top_n=10):
    if weights is None:
        weights = {a: 1 for a in attributes}
    weight_vector = np.array([weights[a] for a in attributes])

    # apply the weights to both the players and the user input
    player_data = players[attributes].to_numpy() * weight_vector
    user_point = np.array([user[a] for a in attributes]) * weight_vector

    # find the nearest players with k-NN
    knn = NearestNeighbors(n_neighbors=top_n, metric=method)
    knn.fit(player_data)
    distances, indices = knn.kneighbors([user_point])

    # turn the distance into an easy 0-100 match score
    if method == "euclidean":
        max_distance = np.sqrt((weight_vector ** 2 * (99 ** 2)).sum())
        match = (1 - distances[0] / max_distance) * 100
    else:  # cosine distance is 1 - similarity
        match = (1 - distances[0]) * 100

    result = players.iloc[indices[0]].copy()
    result["match_percent"] = match.round(1)
    return result[["name", "club", "positions"] + attributes + ["match_percent"]]


# test it
recommend(user, method="euclidean", top_n=5)

,name,club,positions,pace,shooting,passing,dribbling,defending,physical,match_percent
1236,R. Orsolini,Bologna,"CAM, RW",80,70,74,81,39,65,99.3
912,Matheus Pereira,Al Hilal,"CAM, RM, RW",79,74,75,80,39,63,97.9
1925,F. Pizzini,Defensa y Justicia,RM,78,69,72,78,40,65,97.7
1501,F. Di Francesco,Empoli,"CF, LW, ST",79,70,73,81,39,60,97.7
1654,R. Del Castillo,Stade Brestois 29,"RM, LM",77,68,72,79,39,65,97.7


## 9. Quick test

A fixed set of six values, and the player names it returns.

In [9]:
test_input = {
    "pace": 85,
    "shooting": 80,
    "passing": 70,
    "dribbling": 88,
    "defending": 35,
    "physical": 70,
}

results = recommend(test_input, method="euclidean", top_n=5)
print(results["name"].tolist())
results

['A. Martial', 'Gabriel Jesus', 'Matheus Cunha', 'W. Zaha', 'D. Machís']


,name,club,positions,pace,shooting,passing,dribbling,defending,physical,match_percent
296,A. Martial,Manchester United,"ST, LM",87,80,73,85,35,68,97.9
168,Gabriel Jesus,Manchester City,ST,84,81,72,86,39,73,97.6
566,Matheus Cunha,Atlético de Madrid,"CAM, LM, ST",85,78,74,84,31,69,97.0
193,W. Zaha,Crystal Palace,"CF, LM, ST",89,77,73,87,34,75,96.8
295,D. Machís,Granada CF,"LM, RM, ST",88,80,75,83,38,70,96.6
